In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels import PanelOLS
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white
import warnings
warnings.filterwarnings('ignore')

手动创建虚拟变量的方式，进行起源国+目的国+时间固定

In [2]:
# 读取数据
model_data_path = '../data/processed/merged_dataset_cleaned.csv'
data = pd.read_csv(model_data_path)

# 添加交互项
def add_interaction_terms(df, spei_lag_var):
    """
    添加 SPEI 滞后变量的交互项。
    
    Parameters:
    df: pd.DataFrame
        数据框
    spei_lag_var: str
        SPEI 滞后变量的列名
    
    Returns:
    pd.DataFrame
        添加交互项后的数据框
    """
    df['spei_ma_interact'] = df[spei_lag_var] * df['o_MA']
    df['spei_bf_interact'] = df[spei_lag_var] * df['border_friction_ij']
    return df

data = add_interaction_terms(data, 'spei_lag1')

# 按照指定顺序调整列顺序并删除
ordered_columns = [
    # ISO 列
    'origin_iso3', 'destination_iso3',
    # 日期相关
    'migration_date', 'year', 'month',
    # 因变量
    'flow', 'log_flow',
    # 重点自变量
    'o_spei', 'spei_lag1', 'spei_lag2', 'spei_lag3', 'o_MA', 'border_friction_ij',
    # 交互项
    'spei_ma_interact', 'spei_bf_interact',
    # 控制变量
    'o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban',
    'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability',
    'control_distwces'
]

# 重新排列列顺序
data = data[ordered_columns]

# 将浮点列转换为 float32 以节省内存
float_cols = data.select_dtypes(include=['float64']).columns
data[float_cols] = data[float_cols].astype('float32')



In [3]:
data['year_month'] = (
    (pd.to_datetime(data['migration_date']).dt.year - 2019) * 12 +
    pd.to_datetime(data['migration_date']).dt.month
)

In [4]:
# 直接覆盖 origin_iso3
data['origin_iso3'] = pd.factorize(data['origin_iso3'])[0]
data['destination_iso3'] = pd.factorize(data['destination_iso3'])[0]

In [5]:
def prepare_basic_regression_data(data, max_categories=None):
    """
    准备基础回归数据，创建虚拟变量
    """
    print("准备回归数据...")
    
    # 选择需要的变量
    base_vars = ['log_flow', 'spei_lag1', 'spei_ma_interact', 'spei_bf_interact', 
                 'o_MA', 'border_friction_ij', 'origin_iso3', 'destination_iso3', 'year_month']
    
    # 添加控制变量
    control_vars = ['o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban',
                   'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability', 'control_distwces']
    
    # 检查哪些控制变量存在
    available_controls = [var for var in control_vars if var in data.columns]
    print(f"可用控制变量: {available_controls}")
    
    all_vars = base_vars + available_controls
    
    # 选择数据并删除缺失值
    data_clean = data[all_vars].dropna().copy()
    print(f"清理后数据量: {len(data_clean):,}")
    
    # 如果指定了最大类别数，进行限制以避免内存问题
    if max_categories:
        print(f"限制每个维度最多 {max_categories} 个类别...")
        
        # 选择最常见的类别
        origin_top = data_clean['origin_iso3'].value_counts().head(max_categories).index
        dest_top = data_clean['destination_iso3'].value_counts().head(max_categories).index
        time_top = data_clean['year_month'].value_counts().head(max_categories).index
        
        data_clean = data_clean[
            (data_clean['origin_iso3'].isin(origin_top)) &
            (data_clean['destination_iso3'].isin(dest_top)) &
            (data_clean['year_month'].isin(time_top))
        ].copy()
        
        print(f"限制后数据量: {len(data_clean):,}")
    
    print(f"固定效应统计:")
    print(f"  起源国数量: {data_clean['origin_iso3'].nunique()}")
    print(f"  目的国数量: {data_clean['destination_iso3'].nunique()}")
    print(f"  时间点数量: {data_clean['year_month'].nunique()}")
    
    return data_clean, available_controls

def create_fixed_effects_dummies(data_clean):
    """
    创建固定效应虚拟变量
    """
    print("创建固定效应虚拟变量...")
    
    # 创建虚拟变量（drop_first=True避免完全共线性）
    origin_dummies = pd.get_dummies(data_clean['origin_iso3'], prefix='FE_origin', drop_first=True)
    dest_dummies = pd.get_dummies(data_clean['destination_iso3'], prefix='FE_dest', drop_first=True)
    time_dummies = pd.get_dummies(data_clean['year_month'], prefix='FE_time', drop_first=True)
    
    print(f"固定效应虚拟变量数量:")
    print(f"  起源国固定效应: {origin_dummies.shape[1]}")
    print(f"  目的国固定效应: {dest_dummies.shape[1]}")
    print(f"  时间固定效应: {time_dummies.shape[1]}")
    print(f"  总固定效应: {origin_dummies.shape[1] + dest_dummies.shape[1] + time_dummies.shape[1]}")
    
    return origin_dummies, dest_dummies, time_dummies

# 准备数据（限制类别数量以避免内存问题）
data_reg, available_controls = prepare_basic_regression_data(data, max_categories=50)
origin_dummies, dest_dummies, time_dummies = create_fixed_effects_dummies(data_reg)

准备回归数据...
可用控制变量: ['o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban', 'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability', 'control_distwces']
清理后数据量: 1,319,786
限制每个维度最多 50 个类别...
限制后数据量: 118,416
固定效应统计:
  起源国数量: 50
  目的国数量: 50
  时间点数量: 48
创建固定效应虚拟变量...
固定效应虚拟变量数量:
  起源国固定效应: 49
  目的国固定效应: 49
  时间固定效应: 47
  总固定效应: 145


In [6]:
# 创建固定效应虚拟变量
def create_fixed_effects_dummies(data_clean):
    """
    创建固定效应虚拟变量
    """
    print("创建固定效应虚拟变量...")
    
    # 创建虚拟变量（drop_first=True避免完全共线性）
    origin_dummies = pd.get_dummies(data_clean['origin_iso3'], prefix='FE_origin', drop_first=True).astype(int)  # 修改点 1
    dest_dummies = pd.get_dummies(data_clean['destination_iso3'], prefix='FE_dest', drop_first=True).astype(int)  # 修改点 1
    time_dummies = pd.get_dummies(data_clean['year_month'], prefix='FE_time', drop_first=True).astype(int)  # 修改点 1
    
    print(f"固定效应虚拟变量数量:")
    print(f"  起源国固定效应: {origin_dummies.shape[1]}")
    print(f"  目的国固定效应: {dest_dummies.shape[1]}")
    print(f"  时间固定效应: {time_dummies.shape[1]}")
    print(f"  总固定效应: {origin_dummies.shape[1] + dest_dummies.shape[1] + time_dummies.shape[1]}")
    
    return origin_dummies, dest_dummies, time_dummies

In [7]:
# 运行模型并提取结果
def run_regression_models(data_reg, origin_dummies, dest_dummies, time_dummies, available_controls):
    """
    运行多个回归模型并提取关键结果
    """
    print("\n" + "="*60)
    print("开始运行回归模型")
    print("="*60)
    
    results = {}
    models = {
        "Model 1": ['spei_lag1'],
        "Model 2": ['spei_lag1', 'o_MA', 'spei_ma_interact'],
        "Model 3": ['spei_lag1', 'border_friction_ij', 'spei_bf_interact'],
        "Model 4": ['spei_lag1', 'o_MA', 'border_friction_ij', 'spei_ma_interact', 'spei_bf_interact']
    }
    
    for model_name, variables in models.items():
        print("\n" + "-"*50)
        print(f"{model_name}: 回归分析")
        print("-" * 50)
        try:
            # 准备解释变量
            X = pd.concat([
                data_reg[variables],  # 主效应和交互项
                data_reg[available_controls],  # 控制变量
                origin_dummies,               # 起源国固定效应
                dest_dummies,                 # 目的国固定效应
                time_dummies                  # 时间固定效应
            ], axis=1)
            X = sm.add_constant(X)
            y = data_reg['log_flow']
            X = X.astype(float)
            
            # 运行回归
            model = sm.OLS(y, X)
            result = model.fit()
            
            # 提取关键结果
            print(result.summary())
            results[model_name] = {
                "spei_lag1": extract_coefficient(result, 'spei_lag1'),
                "spei_ma_interact": extract_coefficient(result, 'spei_ma_interact'),
                "spei_bf_interact": extract_coefficient(result, 'spei_bf_interact'),
                "o_MA": extract_coefficient(result, 'o_MA'),
                "border_friction_ij": extract_coefficient(result, 'border_friction_ij'),
                "N": int(result.nobs),
                "R2": result.rsquared,
                "Adj_R2": result.rsquared_adj,
                "F_stat": result.fvalue,
                "F_pval": result.f_pvalue
            }
        except Exception as e:
            print(f"❌ {model_name} 运行失败: {str(e)}")
    
    print("\n" + "="*60)
    print("回归模型运行完成")
    print("="*60)
    
    return results

# 提取系数及其统计信息
def extract_coefficient(result, variable):
    """
    提取指定变量的系数、标准误和显著性星号
    """
    try:
        coef = result.params[variable]
        se = result.bse[variable]
        pval = result.pvalues[variable]
        if pval <= 0.01:
            sig = "***"
        elif pval <= 0.05:
            sig = "**"
        elif pval <= 0.1:
            sig = "*"
        else:
            sig = ""
        return {"coef": coef, "se": se, "sig": sig}
    except KeyError:
        return {"coef": None, "se": None, "sig": ""}

# 打印结果
def print_results_summary(results):
    """
    打印所有模型的结果摘要
    """
    for model_name, result in results.items():
        print("\n" + "="*60)
        print(f"{model_name} 结果摘要")
        print("="*60)
        print(f"spei_lag1: {result['spei_lag1']['coef']:.4f} ({result['spei_lag1']['se']:.4f}) {result['spei_lag1']['sig']}")
        print(f"spei_ma_interact: {result['spei_ma_interact']['coef']} ({result['spei_ma_interact']['se']}) {result['spei_ma_interact']['sig']}")
        print(f"spei_bf_interact: {result['spei_bf_interact']['coef']} ({result['spei_bf_interact']['se']}) {result['spei_bf_interact']['sig']}")
        print(f"o_MA: {result['o_MA']['coef']} ({result['o_MA']['se']}) {result['o_MA']['sig']}")
        print(f"border_friction_ij: {result['border_friction_ij']['coef']} ({result['border_friction_ij']['se']}) {result['border_friction_ij']['sig']}")
        print(f"观测值数量 (N): {result['N']}")
        print(f"R²: {result['R2']:.4f}")
        print(f"调整R²: {result['Adj_R2']:.4f}")
        print(f"F统计量: {result['F_stat']:.2f}")
        print(f"F检验p值: {result['F_pval']:.4f}")
        print("包含起源国固定效应: ✓")
        print("包含目的国固定效应: ✓")
        print("包含时间固定效应: ✓")
        print("包含控制变量: ✓")



In [8]:
# 调用代码
data_reg, available_controls = prepare_basic_regression_data(data, max_categories=100)
origin_dummies, dest_dummies, time_dummies = create_fixed_effects_dummies(data_reg)
results = run_regression_models(data_reg, origin_dummies, dest_dummies, time_dummies, available_controls)
print_results_summary(results)

准备回归数据...
可用控制变量: ['o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban', 'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability', 'control_distwces']
清理后数据量: 1,319,786
限制每个维度最多 100 个类别...
限制后数据量: 476,016
固定效应统计:
  起源国数量: 100
  目的国数量: 100
  时间点数量: 48
创建固定效应虚拟变量...
固定效应虚拟变量数量:
  起源国固定效应: 99
  目的国固定效应: 99
  时间固定效应: 47
  总固定效应: 245

开始运行回归模型

--------------------------------------------------
Model 1: 回归分析
--------------------------------------------------
                            OLS Regression Results                            
Dep. Variable:               log_flow   R-squared:                       0.393
Model:                            OLS   Adj. R-squared:                  0.392
Method:                 Least Squares   F-statistic:                     1197.
Date:                Thu, 14 Aug 2025   Prob (F-statistic):               0.00
Time:                        08:57:31   Log-Likelihood:            -8.0380e+05
No. Observations:              476016   AIC: